In [85]:
import json
from pathlib import Path
import sys
import numpy as np

# Define the project root (one folder up from /workshop)
project_root = Path("..").resolve()

# Add the project root to sys.path so Jupyter can find your 'src' module
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

# Calculating weights for positions without any data
For the following positions: CAM, Wide Midfielder, Wingback, we have no available data to calculate weights with. This means that we cannot run them through the same process as other positions: pre-processing and PCA. Creating synthetic data for these positions is also a challenging process, and one that may not yield accurate results.<br><br>
Instead, we will adapt the weights of similar positions:
 - For CAM, we will use the weights of the Central Midfielder (CM) position, as they share similar responsibilities on the field.
 - For Wide Midfielder, we will use the weights of the Winger (RW/LW) position, as they both operate on the flanks and have similar roles in attacking.
 - For Wingback, we will use the weights of the Fullback (RB/LB) position, as they both have defensive duties while also contributing to the attack on the wings.<br><br>

We will take in the relevant proxy weights, and adjust them using multipliers to make the weights more relevant to the specific position and the standard tactical expectations for that position. These multipliers will be determined based on the typical responsibilities and contributions of each position on the field. For example, for CAM, we may increase the weights for passing and xT, while for Wingback, we may increase the weights for attacking contributions and and distance covered.<br><br>

This method is composed of a few main steps:
1. Defining the weight multipliers for each position based on their typical responsibilities and contributions on the field.
2. Applying these multipliers to the proxy weights from the similar positions to calculate the adjusted weights for CAM, Wide Midfielder, and Wingback.
3. Re-normalizing the weights to ensure they sum up to 1, which allows for a consistent comparison across all positions.<br><br>

This approach allows us to estimate the weights for these positions in a way that is informed by the data we have for similar positions, while also accounting for the unique characteristics of each position. It is important to note that this method is not perfect and may not capture all the nuances of each position, but it provides a reasonable approximation given the lack of direct data.

### The wingback
**Proxy: Fullback (RB/LB)**<br><br>
Wingbacks run more, dribble more, and defend less than traditional fullbacks. Therefore, we will increase the weights for distance covered, dribbles, and attacking contributions, while decreasing the weights for defensive actions. The specific multipliers will be determined based on the typical responsibilities of wingbacks in modern football tactics.

**Step 1: Define weight multipliers for Wingback**
 - `Goals`: **2.0** (wingbacks should be much more rewarded for goals than traditional fullbacks)
 - `Assists`: **1.5** (similar to goals, wingbacks should be more rewarded for assists than traditional fullbacks)
 - `Shots`: **1.5** (wingbacks should be rewarded more for shots than traditional fullbacks due to their increased attacking involvement)
 - `Shot Accuracy`: **1.5** (wingbacks should have better shot accuracy due to more attacking involvement)
 - `Passes`: **1.0** (wingbacks may have similar passing involvement as fullbacks)
 - `Pass Accuracy`: **1.0** (wingbacks may have similar pass accuracy as fullbacks)
 - `Dribbles`: **1.8** (taking players on is much more important for wingbacks than traditional fullbacks)
 - `Dribble Success Rate`: **1.8** (wingbacks should be taking players on more effectively than traditional fullbacks)
 - `Tackles`: **0.6** (wingbacks have less defensive responsibilities)
 - `Tackle Success Rate`: **0.7** (due to their reduced defensive responsibilities, wingbacks may have a lower tackle success rate than traditional fullbacks)
 - `Offsides`: **1.0** (wingbacks may have similar offsides involvement as fullbacks)
 - `Fouls Committed`: **1.0** (wingbacks may have similar fouls committed as fullbacks)
 - `Possession Won`: **0.6** (there is a much lower defensive burden on wingbacks, so they may win fewer possessions than traditional fullbacks)
 - `Possession Lost`: **0.7** (as wingbacks are involved in more attacking play, they should be punished less for losing possessions)
 - `Distance Covered`: **1.5** (As wingbacks have more attacking responsibilities, they should be covering more distance than traditional fullbacks)
 - `Distance Sprinted`: **1.5** (similar to distance covered, wingbacks should be sprinting more than traditional fullbacks due to their increased involvement in attacking plays)
 - `xT`: **1.8** (wingbacks should be contributing more to expected threat than traditional fullbacks due to their increased attacking involvement)

In [86]:
wb_multipliers = {
    'goals_p90': 2.0,
    'assists_p90': 2.0,
    'shots_p90': 1.5,
    'shot_accuracy': 1.5,
    'passes_p90': 1.0,
    'pass_accuracy': 1.0,
    'dribbles_p90': 1.8,
    'dribble_success_rate': 1.8,
    'tackles_p90': 0.6,
    'tackle_success_rate': 0.6,
    'offsides_p90': 1.0,
    'fouls_committed_p90': 1.0,
    'possession_won_p90': 0.6,
    'possession_lost_p90': 0.7,
    'distance_covered_p90': 1.5,
    'distance_sprinted_p90': 1.5,
    'xT_bonus_p90': 1.8
}

**Step 2: Apply multipliers to proxy weights and re-normalizing**<br><br>
We will take the weights from the Fullback position and apply the defined multipliers to calculate the adjusted weights for the Wingback position. After applying the multipliers, we will re-normalize the weights to ensure they sum up to 1.

In [87]:
# Importing fullback weights
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["RB"]
fullback_weights = {col: weights_dict[col] for col in col_names}

In [88]:
wingback_weights = {}
total_wb_weight = 0

for stat, base_weight in fullback_weights.items():
    multiplier = wb_multipliers.get(stat, 1.0) # Default to 1.0 if not listed
    raw_new_weight = base_weight * multiplier
    wingback_weights[stat] = raw_new_weight
    total_wb_weight += raw_new_weight

# Re-normalize so they sum to 1.0
for value in wingback_weights.values():
    value /= total_wb_weight

In [89]:
print("Wingback Weights:")
for stat, weight in wingback_weights.items():
    print(f"{stat}: {weight:.4f}")

Wingback Weights:
goals_p90: 0.0000
assists_p90: 0.0163
shots_p90: 0.0110
shot_accuracy: 0.0000
passes_p90: 0.1021
pass_accuracy: 0.0838
dribbles_p90: 0.1551
dribble_success_rate: 0.1043
tackles_p90: 0.0664
tackle_success_rate: 0.0504
offsides_p90: 0.0004
fouls_committed_p90: 0.0587
possession_won_p90: 0.0705
possession_lost_p90: 0.0529
distance_covered_p90: 0.1057
distance_sprinted_p90: 0.1037
xT_bonus_p90: 0.1224


In [90]:
# Saving these weights back to the position_weights.json file
section_names = ["RWB", "LWB"]
weights_dict = dict(zip(col_names, wingback_weights))
position_weights_path = project_root / "workshop" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

for section_name in section_names:
    position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

## The Wide Midfielder
**Proxy: Winger (RW/LW)**<br><br>

Wide Midfielders shoot way less than Wingers, cross more, and actually have to track back and defend. Therefore, we will decrease the weights for goals and shots, increase the weights for crossing and defensive actions, and adjust the weights for passing and dribbling to reflect the different responsibilities of Wide Midfielders compared to Wingers. The specific multipliers will be determined based on the typical responsibilities of Wide Midfielders in modern football

**Step 1: Define weight multipliers for Wide Midfielder**
 - `Goals`: **0.8** (Wide Midfielders shouldn't be expected to score as much as Wingers, as they often play deeper and have more defensive responsibilities)
 - `Assists`: **1.2** (Wide Midfielders should be expected to contribute creatively a bit more than wingers, as they often have more involvement in build-up play and crossing)
 - `Shots`: **0.5** (Wide Midfielders should be expected to take a lot less shots than wingers)
 - `Shot Accuracy`: **0.5** (Wide Midfielders should be expected to have a lower shot accuracy than wingers, as they take fewer shots and often from less dangerous positions)
 - `Passes`: **1.5** (Wide Midfielders should be expected to have more passing involvement than Wingers, as they often play deeper and are more involved in build-up play)
 - `Pass Accuracy`: **1.5** (Wide Midfielders should be expected to have a higher pass accuracy than Wingers, as they are often making their passes from deeper positions and may have more time on the ball)
 - `Dribbles`: **0.8** (Wide Midfielders should be expected to dribble less than wingers, less hero balling and more efficient dribbling)
 - `Dribble Success Rate`: **1.0** (Wide Midfielders may have a similar dribble success rate as Wingers, but this can vary greatly depending on the player's style and role)
 - `Tackles`: **8.0** (Wide Midfielders are typically expected to track back and help out their fullback much more than wingers, so they should be expected to make more tackles)
 - `Tackle Success Rate`: **6.0** (Wide Midfielders should be expected to have a higher tackle success rate than Wingers, as they are often making more tackles and may be more defensively focused)
 - `Offsides`: **1.0** (Wide Midfielders may have similar offsides involvement as Wingers, but this can vary greatly depending on the player's role and style)
 - `Fouls Committed`: **1.0** (Wide Midfielders may have similar fouls committed as Wingers, but this can vary greatly depending on the player's role and style)
 - `Possession Won`: **7.0** (Wide Midfielders should be expected to win more possessions than Wingers, as they often have more defensive responsibilities)
 - `Possession Lost`: **1.5** (Wide Midfielders should be punished more for losing possession in midfield)
 - `Distance Covered`: **1.1** (As Wide Midfielders have more defensive responsibilities than Wingers, they may be expected to cover slightly more distance than Wingers, but this can vary greatly depending on the player's role and style)
 - `Distance Sprinted`: **1.1** (similar to distance covered, Wide Midfielders may be expected to sprint slightly more than Wingers due to their increased defensive responsibilities, but this can vary greatly depending on the player's role and style)
 - `xT`: **1.3** (A higher reward for contributing expected threat, often from wider or deeper positions)

In [91]:
wm_multipliers = {
    'goals_p90': 0.8,
    'assists_p90': 1.2,
    'shots_p90': 0.5,
    'shot_accuracy': 0.5,
    'passes_p90': 1.5,
    'pass_accuracy': 1.5,
    'dribbles_p90': 0.8,
    'dribble_success_rate': 1.0,
    'tackles_p90': 8.0,
    'tackle_success_rate': 6.0,
    'offsides_p90': 1.0,
    'fouls_committed_p90': 1.0,
    'possession_won_p90': 7.0,
    'possession_lost_p90': 1.5,
    'distance_covered_p90': 1.1,
    'distance_sprinted_p90': 1.1,
    'xT_bonus_p90': 1.3
}

**Step 2: Apply multipliers to proxy weights and re-normalizing**<br><br>
We will take the weights from the Winger position and apply the defined multipliers to calculate the adjusted weights for the Wide Midfielder position. After applying the multipliers, we will re-normalize the weights to ensure they sum up to 1.

In [92]:
# Importing winger weights
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["RW"]
winger_weights = {col: weights_dict[col] for col in col_names}

In [93]:
wide_midfielder_weights = {}
total_wm_weight = 0

for stat, base_weight in winger_weights.items():
    multiplier = wm_multipliers.get(stat, 1.0) # Default to 1.0 if not listed
    raw_new_weight = base_weight * multiplier
    wide_midfielder_weights[stat] = raw_new_weight
    total_wm_weight += raw_new_weight

# Re-normalize so they sum to 1.0
for value in wide_midfielder_weights.values():
    value /= total_wm_weight

In [94]:
print("Wide Midfielder Weights:")
for stat, weight in wide_midfielder_weights.items():
    print(f"{stat}: {weight:.4f}")

Wide Midfielder Weights:
goals_p90: 0.0547
assists_p90: 0.0807
shots_p90: 0.0363
shot_accuracy: 0.0363
passes_p90: 0.1336
pass_accuracy: 0.0947
dribbles_p90: 0.0927
dribble_success_rate: 0.1063
tackles_p90: 0.0598
tackle_success_rate: 0.0406
offsides_p90: 0.0444
fouls_committed_p90: 0.0355
possession_won_p90: 0.0654
possession_lost_p90: 0.0393
distance_covered_p90: 0.0801
distance_sprinted_p90: 0.0799
xT_bonus_p90: 0.0906


In [95]:
# Saving these weights back to the position_weights.json file
section_names = ["RM", "LM"]
weights_dict = dict(zip(col_names, wingback_weights))
position_weights_path = project_root / "workshop" / "position_weights.json"

if position_weights_path.exists():
    with open(position_weights_path, "r") as f:
        position_weights = json.load(f)
else:
    position_weights = {}

for section_name in section_names:
    position_weights[section_name] = weights_dict

with open(position_weights_path, "w") as f:
    json.dump(position_weights, f, indent=4)

## The CAM
**Proxy: Central Midfielder (CM)**<br><br>
The CAM is pure creation and goal threat, with much less defensive responsibility than a traditional central midfielder. Therefore, we will increase the weights for goals, assists, passing, and xT, while decreasing the weights for defensive actions and distance covered. The specific multipliers will be determined based on the typical responsibilities of CAMs in modern football tactics.

**Step 1: Define weight multipliers for Wide Midfielder**
 - `Goals`: **2.0** (CAMs should be much more rewarded for goals than traditional central midfielders, as they are often a team's primary goal threat)
 - `Assists`: **2.5** (similar to goals, CAMs should be more rewarded for assists than traditional central midfielders, as they are often a team's primary creative force)
 - `Shots`: **1.5** (CAMs should be rewarded more for shots than traditional central midfielders due to their increased attacking involvement)
 - `Shot Accuracy`: **1.5** (CAMs should have better shot accuracy due to more attacking involvement)
 - `Passes`: **1.5** (CAMs should have more passing involvement than traditional central midfielders, as they are often heavily involved in build-up play)
 - `Pass Accuracy`: **1.2** (CAMs should have a higher pass accuracy than traditional central midfielders, as they are often making more key passes and may have more time on the ball)
 - `Dribbles`: **1.5** (CAMs should be expected to dribble more than traditional central midfielders, as they often have more freedom to take on players and create chances)
 - `Dribble Success Rate`: **1.2** (CAMs should be rewarded more for successful dribbles than traditional central midfielders, as their dribbles are often more impactful and in tighter spaces)
 - `Tackles`: **0.2** (CAMs have less defensive responsibilities than traditional central midfielders, so they should be expected to make fewer tackles)
 - `Tackle Success Rate`: **0.2** (due to their reduced defensive responsibilities, CAMs may have a much lower tackle success rate than traditional central midfielders)
 - `Offsides`: **1.0** (CAMs may have similar offsides involvement as traditional central midfielders, but this can vary greatly depending on the player's role and style)
 - `Fouls Committed`: **1.0** (CAMs may have similar fouls committed as traditional central midfielders, but this can vary greatly depending on the player's role and style)
 - `Possession Won`: **0.3** (there is a much lower defensive burden on CAMs, so they may win significantly fewer possessions than traditional central midfielders)
 - `Possession Lost`: **0.5** (CAMs should have the freedom to be more creative and take more risks, so they should be punished less for losing possessions than traditional central midfielders)
 - `Distance Covered`: **0.8** (As CAMs have less defensive responsibilities than traditional central midfielders, they may be expected to cover less distance than traditional central midfielders, but this can vary greatly depending on the player's role and style)
 - `Distance Sprinted`: **0.8** (similar to distance covered, CAMs may be expected to sprint less than traditional central midfielders due to their reduced defensive responsibilities, but this can vary greatly depending on the player's role and style)
 - `xT`: **2.0** (CAMs should be contributing much more to expected threat than traditional central midfielders due to their increased attacking involvement)

In [96]:
cam_multipliers = {
    'goals_p90': 2.0,
    'assists_p90': 2.5,
    'shots_p90': 1.5,
    'shot_accuracy_p90': 1.5,
    'passes_p90': 1.5,
    'pass_accuracy_p90': 1.2,
    'dribbles_p90': 1.5,
    'dribble_success_rate_p90': 1.2,
    'tackles_p90': 0.2,
    'tackle_success_rate_p90': 0.2,
    'offsides_p90': 1.0,
    'fouls_committed_p90': 1.0,
    'possession_won_p90': 0.3,
    'possession_lost_p90': 0.5,
    'distance_covered_p90': 0.8,
    'distance_sprinted_p90': 0.8,
    'xt_p90': 2.0
}

In [97]:
# Importing central midfielder weights
col_names = ["goals_p90", "assists_p90", "shots_p90", "shot_accuracy", "passes_p90", "pass_accuracy",
              "dribbles_p90", "dribble_success_rate", "tackles_p90", "tackle_success_rate", "offsides_p90",
              "fouls_committed_p90", "possession_won_p90", "possession_lost_p90", "distance_covered_p90", "distance_sprinted_p90", "xT_bonus_p90"]
position_weights_path = project_root / "workshop" / "position_weights.json"
with open(position_weights_path, "r") as f:
    position_weights = json.load(f)
weights_dict = position_weights["CM"]
cm_weights = {col: weights_dict[col] for col in col_names}

In [98]:
cam_weights = {}
total_cam_weight = 0

for stat, base_weight in cm_weights.items():
    multiplier = cam_multipliers.get(stat, 1.0)
    raw_new_weight = base_weight * multiplier
    cam_weights[stat] = raw_new_weight
    total_cam_weight += raw_new_weight

for value in cam_weights.values():
    value /= total_cam_weight

In [99]:
print("Attacking Midfielder Weights:")
for stat, weight in cam_weights.items():
    print(f"{stat}: {weight:.4f}")

Attacking Midfielder Weights:
goals_p90: 0.0833
assists_p90: 0.1088
shots_p90: 0.0673
shot_accuracy: 0.0312
passes_p90: 0.1444
pass_accuracy: 0.0858
dribbles_p90: 0.1026
dribble_success_rate: 0.0830
tackles_p90: 0.0115
tackle_success_rate: 0.0609
offsides_p90: 0.0007
fouls_committed_p90: 0.0587
possession_won_p90: 0.0176
possession_lost_p90: 0.0449
distance_covered_p90: 0.0416
distance_sprinted_p90: 0.0507
xT_bonus_p90: 0.0636
